# F1 · Scaling and metric: what really matters, on three datasets

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Systematically compare three treatments of the scales (none, `StandardScaler`, `MinMaxScaler`) and four metrics (Euclidean, Manhattan, Chebyshev, cosine) on three datasets (wines, breast tumors, handwritten digits) with
a repeated cross-validation, and establish **which choice weighs most** in each case.

## Dataset

`load_wine` (13 features, very different scales), `load_breast_cancer` (30 features, different scales) and `load_digits` (64 pixels, **all on the same scale** from 0 to 16). Model: `KNeighborsClassifier(7, algorithm="brute", metric=...)`.
Validation: `RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)`, accuracy.

## Educational purpose

Replace the rule "always scale" with a precise understanding: when scaling is decisive (different scales), when it is neutral or harmful (already homogeneous scales, where standardizing amplifies the noise of nearly constant pixels) and how much changing the metric is worth.

## Task

1. Build `table`: a `DataFrame` with a two-level index `(dataset, scaling)` (9 rows: 3 datasets × 3 treatments `none`, `standard`, `minmax`) and as columns the four metrics `euclidean`, `manhattan`, `chebyshev`, `cosine` with the mean cross-validated accuracy. Use a `Pipeline` with the scaler (or without) and the KNN.
2. For each dataset save in `best` (dictionary) the pair `(scaling, metric)` with the maximum accuracy.
3. Save `scaling_gain` (dictionary per dataset): best accuracy with a scaler (`standard` or `minmax`) minus the best accuracy without scaling (`none`), over all the metrics.
4. Save `metric_spread` (dictionary per dataset): for the `standard` scaling, difference between the best and the worst metric. Then `dominant_factor` (dictionary per dataset with value `'scaling'` if `scaling_gain > metric_spread`, otherwise `'metric'`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

datasets = {"wine": load_wine(return_X_y=True), "cancer": load_breast_cancer(return_X_y=True), "digits": load_digits(return_X_y=True)}
scalers = {"none": None, "standard": StandardScaler, "minmax": MinMaxScaler}
metrics = ["euclidean", "manhattan", "chebyshev", "cosine"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=0)

## Your work

In [ ]:
index = pd.MultiIndex.from_product([datasets, scalers], names=["dataset", "scaling"])
table = pd.DataFrame(index=index, columns=metrics, dtype=float)
for (name, sc) in table.index:
    X, y = datasets[name]
    for metric in metrics:
        ...
table.round(3)

In [ ]:
best = {}
scaling_gain = {}
metric_spread = {}
dominant_factor = {}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`steps = [] if sc == "none" else [scalers[sc]()]`; `m = make_pipeline(*steps, KNeighborsClassifier(7, algorithm="brute", metric=metric))`; `cross_val_score(m, X, y, cv=cv).mean()`. Best: `table.loc[name].stack().idxmax()`. Digits: the table has `none`, `standard`, `minmax`: compare with `table.loc[name]`.'):
    check(table.shape == (9, 4) and table.notna().all().all() and list(table.columns) == metrics, "9 rows (3 datasets x 3 scalings) and 4 metrics, with no gaps", "one accuracy for every combination")
    check(table.loc[("wine", "none")].max() < 0.82 and table.loc[("wine", "standard")].min() > 0.92, "wine: without scaling below 0.82; with StandardScaler above 0.92 with any metric", "the scales of the wines are very different")
    check(scaling_gain["wine"] > 0.15 and 0.005 < scaling_gain["cancer"] < 0.08, "gain of scaling: over 0.15 on the wines, between 0.005 and 0.08 on the cancer data", "best with scaler minus best without")
    check(scaling_gain["digits"] < 0.005 and table.loc[("digits", "standard"), "euclidean"] < table.loc[("digits", "none"), "euclidean"], "on the digits scaling does not help (gain below 0.005) and StandardScaler worsens the Euclidean", "the pixels already have the same scale")
    check(best["wine"][0] in ("standard", "minmax") and best["digits"][0] in ("none", "minmax"), "the best treatment: a scaler on the wines; none (or minmax) on the digits", "idxmax of the table")
    check(metric_spread["wine"] < 0.06 and scaling_gain["wine"] > 4 * metric_spread["wine"], "on the wines the metric changes less than 0.06 and scaling weighs over 4 times more", "spread = max - min across metrics with StandardScaler")
    check(dominant_factor["wine"] == "scaling" and dominant_factor["digits"] == "metric", "the dominant factor: scaling for the wines, metric for the digits (on the cancer data the two weigh about the same)", "compare scaling_gain and metric_spread")
done()

---
The commented solution is in `solutions/F_compare/F1_scaling_and_metric_on_three_datasets.ipynb`. Open it after you pass the check, or after you have really tried.